# 10 · Tracking the Trajectory

### Recap & why now
Nine notebooks of planning, and no drone has flown yet. This one closes the loop: the
trajectory goes to Project 5's cascade, unchanged, and we find out how much of the
planner's promise survives contact with a real vehicle.

The interface is the one Notebook 01 set up — `ref(t)` returning position, velocity and
acceleration — and nothing downstream needs to know that those three vectors now come from
a quadratic program.

### Learning objectives
1. Hand a planned trajectory to the cascade and measure the tracking.
2. Show what the **feedforward** derivatives are worth, by removing them.
3. Compare minimum snap against Project 5's stop-at-every-waypoint reference.
4. Test **disturbance rejection**, and see where the recovery margin came from.
5. State honestly what this planner does not do.

In [ ]:
# === Standard setup used throughout this notebook ========================
import numpy as np                 # NumPy = fast vector/matrix math, so we never hand-write loops for arithmetic.
import matplotlib.pyplot as plt     # Matplotlib is our plotting engine for every static figure below.
from matplotlib import animation   # Turns a list of frames into a playable movie (used for the animations).
from mpl_toolkits.mplot3d import Axes3D   # Registers the '3d' projection used by the 3-D figures.
from IPython.display import HTML    # Embeds an animation as a self-contained JS player (no ffmpeg required).

%matplotlib inline
# Render animations as an in-browser JavaScript player so they always play, on any machine.
plt.rcParams["animation.html"] = "jshtml"
# Raise the embed size cap (MB) so longer clips are not silently cut off.
plt.rcParams["animation.embed_limit"] = 60
# One consistent, readable look for every figure in the manual.
plt.rcParams.update({"figure.dpi": 80, "font.size": 11, "axes.grid": True})
# Print numbers with 4 decimals and no scientific notation, so output is easy to eyeball.
np.set_printoptions(precision=4, suppress=True)
print("Setup complete — NumPy", np.__version__, "| Matplotlib", plt.matplotlib.__version__)

In [ ]:
# === Polynomial toolkit, built up over Notebooks 02-05 ===================

def poly_val(c, t, der=0):
    """Value of the polynomial c at time t, or of its `der`-th derivative."""
    out = 0.0
    for i in range(der, len(c)):                   # Terms below `der` differentiate away to zero.
        factor = 1.0
        for k in range(der):
            factor *= (i - k)                      # i(i-1)...(i-der+1), the falling factorial.
        out += c[i]*factor*t**(i - der)
    return out

def deriv_row(n, t, der):
    """Row r with r @ c = the der-th derivative at time t. One CONSTRAINT is one row."""
    r = np.zeros(n)
    for i in range(der, n):
        factor = 1.0
        for k in range(der):
            factor *= (i - k)
        r[i] = factor*t**(i - der)
    return r

def cost_matrix(n, T, der=4):
    """Q with c^T Q c = integral from 0 to T of (der-th derivative)^2 dt."""
    Q = np.zeros((n, n))
    for i in range(der, n):
        for j in range(der, n):
            ci = np.prod([i - k for k in range(der)])
            cj = np.prod([j - k for k in range(der)])
            power = i + j - 2*der + 1               # From integrating t^(i-der) * t^(j-der).
            Q[i, j] = ci*cj*T**power/power
    return Q

NCOEF = 8                                          # Order 7: eight coefficients, eight boundary conditions.
g = 9.81                                           # Gravity, needed whenever we turn acceleration into tilt.
print("polynomial toolkit ready — order %d, %d coefficients per segment per axis" % (NCOEF-1, NCOEF))

In [ ]:
# === The multi-segment solver from Notebook 06 ===========================

def seg_row(N, seg, t, der):
    """A row of the big constraint matrix that touches only segment `seg`."""
    r = np.zeros(N)
    r[seg*NCOEF:(seg+1)*NCOEF] = deriv_row(NCOEF, t, der)
    return r

def build_cost(times, der=4):
    """Block-diagonal Q: one cost_matrix per segment, stacked along the diagonal."""
    Q = np.zeros((len(times)*NCOEF, len(times)*NCOEF))
    for s, T in enumerate(times):
        Q[s*NCOEF:(s+1)*NCOEF, s*NCOEF:(s+1)*NCOEF] = cost_matrix(NCOEF, T, der)
    return Q

def build_constraints(waypoints, times):
    """Waypoints, rest at both ends, and continuity of velocity/acceleration/jerk at each join."""
    m = len(times); N = m*NCOEF
    rows, vals = [], []
    for s in range(m):                             # Every segment starts and ends on its waypoints.
        rows.append(seg_row(N, s, 0.0, 0));      vals.append(waypoints[s])
        rows.append(seg_row(N, s, times[s], 0)); vals.append(waypoints[s+1])
    for der in (1, 2, 3):                          # At rest, in every sense, at both ends.
        rows.append(seg_row(N, 0, 0.0, der));          vals.append(0.0)
        rows.append(seg_row(N, m-1, times[m-1], der)); vals.append(0.0)
    for s in range(m - 1):                         # The two sides of each join must AGREE...
        for der in (1, 2, 3):
            rows.append(seg_row(N, s, times[s], der) - seg_row(N, s+1, 0.0, der))
            vals.append(0.0)                       # ...but we never say WHAT they agree on.
    return np.array(rows), np.array(vals)

def solve_min_snap_1d(waypoints, times, der=4):
    """Equality-constrained QP, solved through the KKT system. One axis."""
    Q = build_cost(times, der)
    A, b = build_constraints(waypoints, times)
    KKT = np.block([[2*Q, A.T], [A, np.zeros((len(b), len(b)))]])
    sol = np.linalg.solve(KKT, np.concatenate([np.zeros(Q.shape[0]), b]))
    return sol[:Q.shape[0]].reshape(len(times), NCOEF)      # Drop the Lagrange multipliers.

def sample(coeffs, times, t, der=0):
    """Evaluate the piecewise polynomial at global time t."""
    edges = np.concatenate([[0.0], np.cumsum(times)])
    if t <= 0:         return poly_val(coeffs[0], 0.0, der)
    if t >= edges[-1]: return poly_val(coeffs[-1], times[-1], der)
    s = int(np.searchsorted(edges, t, side="right") - 1)
    return poly_val(coeffs[s], t - edges[s], der)

def min_snap_3d(waypoints, times):
    """Solve each axis separately and wrap the result in the ref(t) interface the cascade wants."""
    W = np.asarray(waypoints, float)
    coeffs = [solve_min_snap_1d(W[:, axis], times) for axis in range(3)]
    total = float(np.sum(times))
    def ref(t):
        t = min(max(t, 0.0), total)
        p = np.array([sample(coeffs[a], times, t, 0) for a in range(3)])
        v = np.array([sample(coeffs[a], times, t, 1) for a in range(3)])
        acc = np.array([sample(coeffs[a], times, t, 2) for a in range(3)])
        if t >= total:
            v = np.zeros(3); acc = np.zeros(3)     # Hold position once the trajectory is finished.
        return p, v, acc
    return ref, total, coeffs

ROUTE = np.array([(0, 0, 0), (0, 0, 1.5), (2.0, 0, 1.5), (2.0, 2.0, 1.5), (2.0, 2.0, 2.5), (0, 0, 1.5)])
DURATIONS = [2.5, 3.0, 3.0, 2.0, 4.0]
g = 9.81
print("solver ready — the standing route has %d waypoints and %d segments, %.1f s total" %
      (len(ROUTE), len(DURATIONS), sum(DURATIONS)))

In [ ]:
# === The vehicle and cascade from Project 5, condensed ===================
def quat_normalize(q): q = np.asarray(q, float); return q/np.linalg.norm(q)
def quat_multiply(a, b):
    aw, ax, ay, az = a; bw, bx, by, bz = b
    return np.array([aw*bw-ax*bx-ay*by-az*bz, aw*bx+ax*bw+ay*bz-az*by,
                     aw*by-ax*bz+ay*bw+az*bx, aw*bz+ax*by-ay*bx+az*bw])
def quat_conjugate(q): return np.array([q[0], -q[1], -q[2], -q[3]])
def quat_to_rotmat(q):
    w, x, y, z = quat_normalize(q)
    return np.array([[1-2*(y*y+z*z), 2*(x*y-w*z), 2*(x*z+w*y)],
                     [2*(x*y+w*z), 1-2*(x*x+z*z), 2*(y*z-w*x)],
                     [2*(x*z-w*y), 2*(y*z+w*x), 1-2*(x*x+y*y)]])
def quat_from_rotmat(R):
    tr = np.trace(R)
    if tr > 0:
        s_ = np.sqrt(tr+1.0)*2
        q = np.array([0.25*s_, (R[2,1]-R[1,2])/s_, (R[0,2]-R[2,0])/s_, (R[1,0]-R[0,1])/s_])
    elif R[0,0] > R[1,1] and R[0,0] > R[2,2]:
        s_ = np.sqrt(1.0+R[0,0]-R[1,1]-R[2,2])*2
        q = np.array([(R[2,1]-R[1,2])/s_, 0.25*s_, (R[0,1]+R[1,0])/s_, (R[0,2]+R[2,0])/s_])
    elif R[1,1] > R[2,2]:
        s_ = np.sqrt(1.0+R[1,1]-R[0,0]-R[2,2])*2
        q = np.array([(R[0,2]-R[2,0])/s_, (R[0,1]+R[1,0])/s_, 0.25*s_, (R[1,2]+R[2,1])/s_])
    else:
        s_ = np.sqrt(1.0+R[2,2]-R[0,0]-R[1,1])*2
        q = np.array([(R[1,0]-R[0,1])/s_, (R[0,2]+R[2,0])/s_, (R[1,2]+R[2,1])/s_, 0.25*s_])
    return quat_normalize(q)
def quat_to_euler(q):
    w, x, y, z = quat_normalize(q)
    return np.array([np.arctan2(2*(w*x+y*z), 1-2*(x*x+y*y)), np.arcsin(np.clip(2*(w*y-z*x), -1, 1)),
                     np.arctan2(2*(w*z+x*y), 1-2*(y*y+z*z))])

PARAMS = dict(m=1.0, L=0.25, I=np.diag([0.01, 0.01, 0.02]), d=0.016, T_min=0.0, T_max=6.0)
ARM = PARAMS["L"]/np.sqrt(2)
MOTOR_POS = np.array([[ARM,-ARM,0.], [ARM,ARM,0.], [-ARM,ARM,0.], [-ARM,-ARM,0.]])
SPIN = np.array([-1., 1., -1., 1.])
MIX = np.vstack([np.ones(4), MOTOR_POS[:,1], -MOTOR_POS[:,0], -SPIN*PARAMS["d"]])
POS, VEL, QUAT, OMEGA = slice(0,3), slice(3,6), slice(6,10), slice(10,13)
GAINS = dict(Kp=np.array([2.,2,3]), Kv=np.array([4.,4,5]), K_R=np.array([12.,12,6]),
             K_w=np.array([.06,.06,.06]), v_max=4.0, tilt_max=np.deg2rad(35))

def quad_dynamics(s, T4, p=PARAMS, f_ext=np.zeros(3)):
    q = quat_normalize(s[QUAT]); w = s[OMEGA]
    T, tx, ty, tz = MIX @ np.asarray(T4, float)
    v_dot = (quat_to_rotmat(q) @ np.array([0,0,T]) + np.array([0,0,-p["m"]*g]) + f_ext)/p["m"]
    return np.concatenate([s[VEL], v_dot, 0.5*quat_multiply(q, np.array([0., *w])),
                           np.linalg.solve(p["I"], np.array([tx,ty,tz]) - np.cross(w, p["I"] @ w))])

def rk4_step(s, T4, dt, p=PARAMS, f_ext=np.zeros(3)):
    k1 = quad_dynamics(s, T4, p, f_ext); k2 = quad_dynamics(s+dt/2*k1, T4, p, f_ext)
    k3 = quad_dynamics(s+dt/2*k2, T4, p, f_ext); k4 = quad_dynamics(s+dt*k3, T4, p, f_ext)
    s2 = s + dt/6*(k1+2*k2+2*k3+k4); s2[QUAT] = quat_normalize(s2[QUAT]); return s2

def cascade(s, p_des, v_ff, a_ff, yaw=0.0, p=PARAMS, K=GAINS):
    """Project 5's six controllers, condensed into one function."""
    v_cmd = K["Kp"]*(p_des - s[POS]) + v_ff
    n_ = np.linalg.norm(v_cmd)
    if n_ > K["v_max"]: v_cmd = v_cmd*K["v_max"]/n_
    a_cmd = K["Kv"]*(v_cmd - s[VEL]) + a_ff
    F = p["m"]*(a_cmd + np.array([0, 0, g])); fz = max(F[2], 0.4*p["m"]*g); fxy = F[:2]
    mx = fz*np.tan(K["tilt_max"])
    if np.linalg.norm(fxy) > mx: fxy = fxy*mx/np.linalg.norm(fxy)
    F = np.array([fxy[0], fxy[1], fz]); T = float(F @ quat_to_rotmat(s[QUAT])[:, 2])
    z_des = F/np.linalg.norm(F); x_c = np.array([np.cos(yaw), np.sin(yaw), 0.])
    y_des = np.cross(z_des, x_c); y_des /= np.linalg.norm(y_des)
    q_des = quat_from_rotmat(np.column_stack([np.cross(y_des, z_des), y_des, z_des]))
    q_e = quat_multiply(quat_conjugate(s[QUAT]), q_des)
    if q_e[0] < 0: q_e = -q_e
    tau = K["K_w"]*(2*K["K_R"]*q_e[1:] - s[OMEGA]) + np.cross(s[OMEGA], p["I"] @ s[OMEGA])
    return np.clip(np.linalg.solve(MIX, np.array([T, *tau])), p["T_min"], p["T_max"])

def fly(ref, T_end, dt=0.005, p=PARAMS, K=GAINS, f_ext=lambda t: np.zeros(3)):
    """Closed-loop flight through the Project 5 cascade."""
    s = np.concatenate([[0,0,0], [0,0,0], [1,0,0,0], [0,0,0]]).astype(float)
    ts, xs, ms, rs = [0.0], [s.copy()], [], []
    for k in range(int(round(T_end/dt))):
        p_des, v_ff, a_ff = ref(k*dt)
        T4 = cascade(s, p_des, v_ff, a_ff, 0.0, p, K)
        s = rk4_step(s, T4, dt, p, f_ext(k*dt))
        ts.append((k+1)*dt); xs.append(s.copy()); ms.append(T4); rs.append(p_des)
    return np.array(ts), np.array(xs), np.array(ms), np.array(rs)

def profile_peaks(ref, total, n=900):
    """Peak speed and peak acceleration a trajectory demands."""
    grid = np.linspace(0, total, n)
    V = np.array([ref(t_)[1] for t_ in grid]); A = np.array([ref(t_)[2] for t_ in grid])
    return np.linalg.norm(V, axis=1).max(), np.linalg.norm(A, axis=1).max()

print("vehicle + cascade loaded — Project 5, unchanged")

## 1 · Flying the plan

The controller is untouched. The only change from Project 5 is that $v_{ff}$ and $a_{ff}$
now carry the trajectory's derivatives instead of zeros.

In [ ]:
ref_snap, T_traj, _ = min_snap_3d(ROUTE, DURATIONS)
t, X, M, R = fly(ref_snap, T_traj + 3.0)
err = np.linalg.norm(X[1:, POS] - R, axis=1)
rpy = np.degrees(np.array([quat_to_euler(q_) for q_ in X[:, QUAT]]))

print("RMS tracking error %.4f m, worst %.4f m at t = %.2f s" %
      (np.sqrt(np.mean(err**2)), err.max(), t[1:][np.argmax(err)]))
print("peak tilt %.1f° against the %.0f° command limit, motors %.2f-%.2f N of %.1f available" %
      (np.abs(rpy[:, :2]).max(), np.degrees(GAINS["tilt_max"]), M.min(), M.max(), PARAMS["T_max"]))
print("motors saturated %.1f%% of the flight" % (100*np.mean(M.max(axis=1) >= PARAMS["T_max"]-1e-9)))

fig = plt.figure(figsize=(13.5, 3.6))
ax = fig.add_subplot(131, projection="3d")
ax.plot(R[:, 0], R[:, 1], R[:, 2], color="C2", ls="--", lw=2, label="planned")
ax.plot(X[:, 0], X[:, 1], X[:, 2], color="C0", lw=1.7, label="flown")
ax.plot(ROUTE[:, 0], ROUTE[:, 1], ROUTE[:, 2], "*", color="k", ms=9)
ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z"); ax.legend(fontsize=8)
ax.set_title("Plan vs flight"); ax.view_init(elev=24, azim=-62)
ax2 = fig.add_subplot(132)
ax2.plot(t[1:], err, color="C3", lw=1.5)
ax2.set_xlabel("time [s]"); ax2.set_ylabel("error [m]"); ax2.set_title("Tracking error")
ax3 = fig.add_subplot(133)
for i in range(4):
    ax3.plot(t[1:], M[:, i], lw=1.0)
ax3.axhline(PARAMS["T_max"], color="C3", ls="--", lw=1.1)
ax3.axhline(PARAMS["m"]*g/4, color="0.6", ls=":", lw=1.1)
ax3.set_xlabel("time [s]"); ax3.set_ylabel("motor thrust [N]")
ax3.set_title("Motors (dotted = hover, dashed = limit)")
plt.tight_layout(); plt.show()

## 2 · What the derivatives are worth

Worth measuring rather than assuming. The cell below flies the identical path twice: once
with $v_d$ and $a_d$ fed forward, once with them zeroed so the controller sees only a
moving setpoint.

In [ ]:
ref_no_ff = lambda t_: (ref_snap(t_)[0], np.zeros(3), np.zeros(3))    # Same path, no derivatives.
t2, X2, M2, R2 = fly(ref_no_ff, T_traj + 3.0)
err2 = np.linalg.norm(X2[1:, POS] - R2, axis=1)

print("                    RMS error    worst error")
print("  with feedforward %11.4f m %13.4f m" % (np.sqrt(np.mean(err**2)), err.max()))
print("  position only    %11.4f m %13.4f m" % (np.sqrt(np.mean(err2**2)), err2.max()))
print("\n  feedforward improves RMS tracking by a factor of %.0f." %
      (np.sqrt(np.mean(err2**2))/np.sqrt(np.mean(err**2))))

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12.5, 3.4))
a1.plot(t[1:], err, color="C0", lw=1.6, label="with feedforward")
a1.plot(t2[1:], err2, color="C3", lw=1.6, label="position only")
a1.set_xlabel("time [s]"); a1.set_ylabel("tracking error [m]"); a1.legend(fontsize=8)
a1.set_title("The cost of not saying what is coming")
a2.plot(R[:, 0], R[:, 1], color="C2", ls="--", lw=2, label="planned")
a2.plot(X[:, 0], X[:, 1], color="C0", lw=1.6, label="with feedforward")
a2.plot(X2[:, 0], X2[:, 1], color="C3", lw=1.4, label="position only")
a2.set_xlabel("x [m]"); a2.set_ylabel("y [m]"); a2.set_aspect("equal"); a2.legend(fontsize=8)
a2.set_title("Top view: corner-cutting without it")
plt.tight_layout(); plt.show()

print("Without feedforward the drone is permanently BEHIND the reference: a proportional")
print("controller needs an error to produce a command, so it can only chase. The feedforward")
print("supplies the velocity and acceleration the path needs, and feedback is left to correct")
print("the small remainder — which is what feedback is good at.")

## 3 · Against Project 5's reference

Project 5 flew the same route with a minimum-jerk, stop-at-every-waypoint reference. Same
waypoints, same total duration, same controller — the only difference is the planner.

In [ ]:
def min_jerk_ref(route, times):
    """Project 5's stop-at-every-waypoint reference, for comparison."""
    route = np.asarray(route, float)
    edges = np.concatenate([[0.0], np.cumsum(times)])
    def ref(t_):
        if t_ <= 0:         return route[0].copy(), np.zeros(3), np.zeros(3)
        if t_ >= edges[-1]: return route[-1].copy(), np.zeros(3), np.zeros(3)
        i = int(np.searchsorted(edges, t_, side="right") - 1)
        T = times[i]; tau = (t_ - edges[i])/T
        s_ = 10*tau**3 - 15*tau**4 + 6*tau**5
        sd = 30*tau**2 - 60*tau**3 + 30*tau**4
        sdd = 60*tau - 180*tau**2 + 120*tau**3
        d_ = route[i+1] - route[i]
        return route[i] + d_*s_, d_*sd/T, d_*sdd/T**2
    return ref, float(edges[-1])

ref_jerk, T_jerk = min_jerk_ref(ROUTE, DURATIONS)
print("%-28s %11s %11s %11s" % ("reference", "RMS error", "worst", "peak tilt"))
for name, ref_, T_ in [("min jerk (stop-and-go)     ", ref_jerk, T_jerk),
                       ("min snap (through)         ", ref_snap, T_traj)]:
    tt, XX, MM, RR = fly(ref_, T_ + 3.0)
    ee = np.linalg.norm(XX[1:, POS] - RR, axis=1)
    rr = np.degrees(np.array([quat_to_euler(q_) for q_ in XX[:, QUAT]]))
    print("%-28s %10.4f m %10.4f m %10.1f°" %
          (name, np.sqrt(np.mean(ee**2)), ee.max(), np.abs(rr[:, :2]).max()))

print("\nSame route, same duration, same controller — and minimum snap tracks better with less")
print("tilt. The vehicle is asked for less, so it delivers more accurately. That is the payoff")
print("for the whole project, and it cost nothing downstream: the controller never changed.")

## 4 · A shove in mid-flight

Feedback earns its keep against things the plan did not predict. A 3 N push — about 30% of
the vehicle's weight — for half a second. All that unused authority from Section 1 is what
pays for the recovery.

In [ ]:
gust = lambda t_: np.array([3.0, 0.0, 0.0]) if 5.0 <= t_ < 5.5 else np.zeros(3)
t_g, X_g, M_g, R_g = fly(ref_snap, T_traj + 3.0, f_ext=gust)
e_g = np.linalg.norm(X_g[1:, POS] - R_g, axis=1)
after = t_g[1:] >= 5.0
peak_t = t_g[1:][after][np.argmax(e_g[after])]
back = (e_g[after] < 0.05) & (t_g[1:][after] > peak_t)

print("undisturbed RMS %.4f m, disturbed RMS %.4f m" %
      (np.sqrt(np.mean(err**2)), np.sqrt(np.mean(e_g**2))))
print("peak error after the push %.3f m at t = %.2f s (the push ended at 5.5 s)" %
      (e_g[after].max(), peak_t))
print("back inside 5 cm at t = %.2f s, %.2f s after the peak" %
      (t_g[1:][after][np.argmax(back)], t_g[1:][after][np.argmax(back)] - peak_t))
print("peak motor during the recovery %.2f N of %.1f — still %.0f%% headroom left" %
      (M_g.max(), PARAMS["T_max"], 100*(1 - M_g.max()/PARAMS["T_max"])))

fig, ax = plt.subplots(figsize=(7.4, 3.0))
ax.plot(t[1:], err, color="0.6", lw=1.4, label="no disturbance")
ax.plot(t_g[1:], e_g, color="C3", lw=1.7, label="3 N push for 0.5 s")
ax.axvspan(5.0, 5.5, color="C1", alpha=0.25)
ax.set_xlabel("time [s]"); ax.set_ylabel("tracking error [m]"); ax.legend(fontsize=9)
ax.set_title("Blown off the path, then back on it")
plt.show()

## 🧪 Try it yourself

**E1.** The planned trajectory used a fraction of the vehicle's tilt and never saturated a
motor. What did that restraint buy, given that a more aggressive plan would also have
arrived?

**E2.** Speed the trajectory up by scaling the times down, and find what breaks first —
tilt, saturation, or tracking error.

In [ ]:
# --- Solution E1 ---
print("E1: headroom, and Section 4 spent it. The undisturbed flight used motors between %.2f and" % M.min())
print("    %.2f N of the %.1f available and %.1f° of the %.0f° tilt limit. The gust then needed" %
      (M.max(), PARAMS["T_max"], np.abs(rpy[:, :2]).max(), np.degrees(GAINS["tilt_max"])))
print("    %.2f N and %.1f°, and there was room for it." %
      (M_g.max(), np.abs(np.degrees(np.array([quat_to_euler(q_) for q_ in X_g[:, QUAT]]))[:, :2]).max()))
print("    A drone flown at its limits has nothing left for the disturbance it did not plan for —")
print("    and disturbances are the only reason feedback exists.")

# --- Solution E2 ---
print("\nE2:  time scale   duration   peak tilt   saturated   RMS error")
for scale in (1.0, 0.7, 0.5, 0.4, 0.3):
    ref_s, T_s, _ = min_snap_3d(ROUTE, [d_*scale for d_ in DURATIONS])
    ts, Xs, Ms, Rs = fly(ref_s, T_s + 3.0)
    es = np.linalg.norm(Xs[1:, POS] - Rs, axis=1)
    rps = np.degrees(np.array([quat_to_euler(q_) for q_ in Xs[:, QUAT]]))
    print("    %8.1fx %10.1f s %10.1f° %10.1f%% %12.4f m" %
          (scale, T_s, np.abs(rps[:, :2]).max(),
           100*np.mean(Ms.max(axis=1) >= PARAMS["T_max"]-1e-9), np.sqrt(np.mean(es**2))))

print("    Peak acceleration scales as 1/T^2, so halving the durations quadruples the tilt demand.")
print("    Tilt runs out first, well before the motors do, because this vehicle's thrust-to-weight")
print("    of %.1f is generous. On a heavier drone the order would reverse." %
      (4*PARAMS["T_max"]/(PARAMS["m"]*g)))
print("    Either way the fix is not a better controller: it is a slower trajectory, and")
print("    Notebook 08's bisection is how you find the fastest one that is still legal.")

## 🚁 Mini-project: the complete flight

Everything this project built, flown by everything Project 5 built: a minimum-snap
trajectory through five waypoints, tracked by a cascaded controller, with the rotors
shaded by load so you can watch the mixer working.

In [ ]:
step = 16
fig = plt.figure(figsize=(7.0, 5.4))
ax = fig.add_subplot(111, projection="3d")

def draw_quad(ax_, position, q, thrusts, scale=2.2):
    """Four arms and four rotors, coloured by how hard each motor is working."""
    R_ = quat_to_rotmat(q)
    for i, mp in enumerate(MOTOR_POS):
        tip = position + R_ @ (mp*scale)
        seg = np.array([position, tip])
        ax_.plot(seg[:, 0], seg[:, 1], seg[:, 2], color="0.35", lw=2)
        load = np.clip(thrusts[i]/PARAMS["T_max"], 0, 1)
        ax_.plot([tip[0]], [tip[1]], [tip[2]], "o", ms=7, color=plt.cm.YlOrRd(0.3 + 0.7*load))
    ax_.quiver(*position, *(R_[:, 2]*0.8), color="C1", lw=2.2, arrow_length_ratio=0.25)

def frame(j):
    ax.clear()
    k = step*j
    ax.plot(R[:, 0], R[:, 1], R[:, 2], color="C2", ls="--", lw=1.5)
    ax.plot(X[:k+1, 0], X[:k+1, 1], X[:k+1, 2], color="C0", lw=1.8)
    draw_quad(ax, X[k, POS], X[k, QUAT], M[min(k, len(M)-1)])
    ax.set_xlim(-0.8, 2.8); ax.set_ylim(-0.8, 2.8); ax.set_zlim(0, 3.2)
    ax.set_box_aspect([3.6, 3.6, 3.2])
    ax.set_xlabel("x — East [m]"); ax.set_ylabel("y — North [m]"); ax.set_zlabel("z — Up [m]")
    ax.set_title("t = %5.2f s   error %5.3f m   tilt %4.1f°" %
                 (k*0.005, np.linalg.norm(X[k, POS] - R[min(k, len(R)-1)]),
                  np.abs(rpy[k, :2]).max()), fontsize=10)
    ax.view_init(elev=24, azim=-62)
    return []

anim = animation.FuncAnimation(fig, frame, frames=len(X)//step, interval=50, blit=False)
plt.close(fig)
HTML(anim.to_jshtml())

## 🤖 What this project built

```text
   01-02   why a step is unanswerable; polynomials and boundary conditions
              │
   03-04   minimum jerk, then the argument for minimum snap
              │
   05-06   one segment as a QP; many segments, joined smoothly
              │
   07      CVXPY, inequality constraints, and infeasibility as information
              │
   08      time allocation — the part the QP cannot solve for you
              │
   09      corridors and potential fields; convex and not
              │
   10      handed to the controller, unchanged
```

> **🤖 Robotics connection.** The gap between this and a research planner is smaller than
> it looks. Safe-flight-corridor methods replace our single tube with a chain of convex
> polyhedra carved out of a voxel map; time allocation gets its own outer optimisation;
> and the whole thing re-solves at a few hertz as new obstacles appear. The structure — a
> convex inner problem, a heuristic outer loop over times, and a geometric certificate of
> safety computed by something else — is the one you have just built.

**The honest limits.** The corridor assumes somebody else already found a safe route;
Notebook 09 placed the detour waypoint by hand. Segment times come from a heuristic plus a
bisection, with no optimality guarantee, because optimising over times is non-convex. And
the planner assumes the drone follows what it asks — which Notebook 08's margin experiment
showed is only true if you plan to a fraction of the vehicle's real limits.